In [ ]:
!export KAGGLE_API_TOKEN=KGAT_8e742fd0b13975e80990e57844537985

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "IMDB Dataset.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
First 5 records:                                               review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import re, random
from pathlib import Path
from typing import List

from transformers import (
    T5TokenizerFast,
    T5ForConditionalGeneration,
    get_linear_schedule_with_warmup
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F


In [ ]:
# Chunking config
SENTS_PER_CHUNK = 3
MIN_SENT_LEN_CHARS = 3
MAX_T5_TOKENS = 144   # cap by T5 token length (match your training MAX_LEN)
SEED = 42
random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

MODEL_NAME = "t5-small"
t5_tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)

MODEL_NAME = "t5-small"
LATENT_DIM = 512
BOTTLENECK_SEQ_LEN = 32
MAX_LEN = 144
BATCH_SIZE = 16
LR = 1e-4
WEIGHT_DECAY = 0.01
EPOCHS = 2
WARMUP_RATIO = 0.03
GRAD_CLIP = 1.0
SEED = 42

# === COPY CURRICULUM / DENOISE RAMP ===
TARGET_NOISE_PROB = 0.15     # final denoising level
NOISE_WARMUP_STEPS = 4000    # 2000–5000 recommended

# === LATENT REGULARIZATION OFF (for now) ===
Z_NOISE_STD = 0.0
Z_DROPOUT_P = 0.0

In [ ]:
def clean_html(text: str) -> str:
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

_ABBR = [
    "Mr", "Mrs", "Ms", "Dr", "Prof", "Sr", "Jr",
    "St", "Mt", "Gen", "Col", "Sgt", "Capt",
    "e.g", "i.e", "etc", "vs",
    "U.S", "U.K", "No", "Dept", "Inc", "Ltd",
]
_ABBR_PATTERN = r"(?:%s)" % "|".join(re.escape(a) for a in _ABBR)

def split_sentences(text: str) -> List[str]:
    text = text.strip()
    if not text:
        return []
    text = text.replace("...", " <ELLIPSIS> ")

    def _protect_abbr(m):
        return m.group(0).replace(".", "<DOT>")

    text = re.sub(rf"\b{_ABBR_PATTERN}\.", _protect_abbr, text)
    text = re.sub(r"\b([A-Z])\.", r"\1<DOT>", text)
    text = re.sub(r"\b([A-Z])<DOT>\s*([A-Z])<DOT>", r"\1<DOT>\2<DOT>", text)

    parts = re.split(r"(?<=[.!?])\s+(?=(?:[\"'(\[])?[A-Z0-9])", text)

    sents = []
    for p in parts:
        p = p.replace("<DOT>", ".").replace("<ELLIPSIS>", "...")
        p = re.sub(r"\s+", " ", p).strip()
        if len(p) >= MIN_SENT_LEN_CHARS:
            sents.append(p)
    return sents

def t5_token_len(text: str) -> int:
    return len(t5_tokenizer.encode(text, add_special_tokens=False))

def split_by_t5_tokens(text: str, max_tokens: int) -> List[str]:
    if t5_token_len(text) <= max_tokens:
        return [text]

    words = text.split()
    out = []
    cur = []
    for w in words:
        trial = " ".join(cur + [w]) if cur else w
        if t5_token_len(trial) <= max_tokens:
            cur.append(w)
        else:
            if cur:
                out.append(" ".join(cur))
            cur = [w]
    if cur:
        out.append(" ".join(cur))

    final = []
    for chunk in out:
        ids = t5_tokenizer.encode(chunk, add_special_tokens=False)
        if len(ids) <= max_tokens:
            final.append(chunk)
        else:
            ids = ids[:max_tokens]
            final.append(t5_tokenizer.decode(ids, skip_special_tokens=True).strip())
    return [c for c in final if c.strip()]

def make_chunks(sentences: List[str], sents_per_chunk: int) -> List[str]:
    chunks = []
    i = 0
    n = len(sentences)
    while i < n:
        base_chunk = " ".join(sentences[i:i + sents_per_chunk]).strip()
        if base_chunk:
            chunks.extend(split_by_t5_tokens(base_chunk, MAX_T5_TOKENS))
        i += sents_per_chunk
    return chunks

In [ ]:


class BottleneckT5AE(nn.Module):
    def __init__(self, model_name: str, latent_dim: int, bottleneck_seq_len: int, pool_slots: int = 16):
        super().__init__()
        self.t5 = T5ForConditionalGeneration.from_pretrained(model_name)
        d_model = self.t5.config.d_model

        self.latent_dim = latent_dim
        self.bottleneck_seq_len = bottleneck_seq_len
        self.pool_slots = pool_slots
        self.d_model = d_model

        self.pool_queries = nn.Parameter(torch.randn(pool_slots, d_model) * 0.02)
        self.pool_q_proj = nn.Linear(d_model, d_model, bias=False)
        self.pool_k_proj = nn.Linear(d_model, d_model, bias=False)
        self.pool_v_proj = nn.Linear(d_model, d_model, bias=False)

        self.down = nn.Linear(d_model, latent_dim)
        self.up = nn.Linear(latent_dim, d_model)

        self.to_mem = nn.Linear(pool_slots * d_model, bottleneck_seq_len * d_model)
        self.mem_slot_emb = nn.Parameter(torch.randn(bottleneck_seq_len, d_model) * 0.02)

    def pool_slots_from_encoder(self, h: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B, L, d = h.shape
        K = self.pool_slots

        Q = self.pool_q_proj(self.pool_queries)[None, :, :].expand(B, K, d)
        Kh = self.pool_k_proj(h)
        Vh = self.pool_v_proj(h)

        scores = torch.einsum("bkd,bld->bkl", Q, Kh) / (d ** 0.5)
        scores = scores.float()
        neg_inf = torch.finfo(scores.dtype).min
        scores = scores.masked_fill(mask[:, None, :] == 0, neg_inf)
        w = torch.softmax(scores, dim=-1).to(Vh.dtype)
        pooled = torch.einsum("bkl,bld->bkd", w, Vh)
        return pooled

    def encode_to_mem(self, input_ids, attention_mask):
        enc_out_full = self.t5.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = enc_out_full.last_hidden_state  # [B,L,d]

        pooled = self.pool_slots_from_encoder(h, attention_mask)  # [B,K,d]
        z_slots = self.down(pooled)  # [B,K,latent]

        if self.training and Z_DROPOUT_P > 0:
            drop_mask = (torch.rand_like(z_slots) > Z_DROPOUT_P).to(z_slots.dtype)
            z_slots = z_slots * drop_mask

        if self.training and Z_NOISE_STD > 0:
            z_slots = z_slots + Z_NOISE_STD * torch.randn_like(z_slots)

        pooled_rec = self.up(z_slots)       # [B,K,d]
        flat = pooled_rec.reshape(pooled_rec.size(0), -1)        # [B,K*d]
        mem_flat = self.to_mem(flat)                             # [B,M*d]
        mem = mem_flat.view(mem_flat.size(0), self.bottleneck_seq_len, self.d_model)  # [B,M,d]

        mem = mem + self.mem_slot_emb[None, :, :]
        mem_mask = torch.ones(mem.size(0), mem.size(1), dtype=torch.long, device=mem.device)
        return mem, mem_mask, z_slots

    def forward(self, input_ids, attention_mask, labels=None):
        mem, mem_mask, _ = self.encode_to_mem(input_ids, attention_mask)
        enc_out = BaseModelOutput(last_hidden_state=mem)
        out = self.t5(
            encoder_outputs=enc_out,
            attention_mask=mem_mask,
            labels=labels
        )
        return out


In [ ]:
def sentiment_to_label(s):
    return 1 if str(s).lower().startswith("pos") else 0

def review_to_chunks(review_text: str):
    t = clean_html(str(review_text))
    sents = split_sentences(t)
    chunks = make_chunks(sents, SENTS_PER_CHUNK)  # already token-capped
    return chunks

@torch.no_grad()
def encode_chunks_to_zmean(ae_model, tokenizer, chunks, device, max_len=144):
    ae_model.eval()
    enc = tokenizer(
        chunks,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_len
    ).to(device)

    _, _, z_slots = ae_model.encode_to_mem(enc["input_ids"], enc["attention_mask"])  # [N,K,latent]
    z_mean = z_slots.mean(dim=1)  # [N,latent]
    z_mean = F.normalize(z_mean.float(), dim=-1)
    return z_mean


In [ ]:
class ChunkMemRNN(nn.Module):
    def __init__(self, latent_dim=512, hidden_dim=256, topk=4, num_classes=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.topk = topk

        self.Wq = nn.Linear(hidden_dim, latent_dim, bias=False)
        self.Wg = nn.Linear(hidden_dim + latent_dim, latent_dim)

        # <I FIXED THIS> make the recurrence actually use h_{t-1} (matches your stated equation)
        self.Wx = nn.Linear(latent_dim, hidden_dim, bias=False)   # x_t -> hidden
        self.Wh = nn.Linear(hidden_dim, hidden_dim, bias=False)   # h_{t-1} -> hidden
        self.Wm = nn.Linear(latent_dim, hidden_dim, bias=False)   # memory -> hidden

        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, M):
        # M: [N,D] normalized, single review
        device = M.device
        N, D = M.shape
        h = torch.zeros(self.hidden_dim, device=device)

        gate_means = []
        entropies = []

        for t in range(N):
            x_t = M[t]

            q = F.normalize(self.Wq(h), dim=-1)  # [D]

            # <I FIXED THIS> causal retrieval: only retrieve from PAST chunks (prevents "future leak" cheating)
            if t == 0:
                c = torch.zeros(D, device=device)
                w = None
            else:
                Mpast = M[:t]                   # only previous chunks
                scores = (Mpast @ q)            # [t]

                k = min(self.topk, t)
                topv, topi = torch.topk(scores, k=k, dim=0)
                w = torch.softmax(topv, dim=0)  # [k]
                Cr = Mpast[topi]                # [k,D]
                c = (w[:, None] * Cr).sum(dim=0)

            # <I FIXED THIS> entropy should be well-defined even at t==0
            if w is None:
                entropies.append(torch.tensor(0.0, device=device))
            else:
                entropies.append(-(w * (w + 1e-8).log()).sum())

            g = torch.sigmoid(self.Wg(torch.cat([h, c], dim=0)))  # [D]
            gate_means.append(g.mean())

            # <I FIXED THIS> true RNN-style update: Wx*x_t + Wh*h_{t-1} + Wm*(g*c)
            h = torch.tanh(self.Wx(x_t) + self.Wh(h) + self.Wm(g * c))

        logits = self.classifier(h)  # [C]
        aux = {
            "gate_mean": torch.stack(gate_means).mean() if gate_means else torch.tensor(0., device=device),
            "attn_entropy": torch.stack(entropies).mean() if entropies else torch.tensor(0., device=device),
        }
        return logits, aux



In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, df):
        self.texts = df["review"].astype(str).tolist()
        self.labels = torch.tensor([sentiment_to_label(s) for s in df["sentiment"]], dtype=torch.long)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_reviews(batch):
    texts, labels = zip(*batch)
    return list(texts), torch.stack(list(labels), dim=0)


In [ ]:
def train_epoch(memrnn, ae_model, tokenizer, loader, opt, device,
                max_chunks=16, max_len=144,
                lam_gate=0.05, tau=0.20, lam_ent=0.01):
    memrnn.train()
    total_loss = 0.0
    n = 0

    for reviews, y in loader:
        y = y.to(device)
        opt.zero_grad(set_to_none=True)

        loss_sum = 0.0
        used = 0

        for review, yi in zip(reviews, y):
            chunks = review_to_chunks(review)
            if len(chunks) == 0:
                continue

            if len(chunks) > max_chunks:
                chunks = chunks[:max_chunks]

            M = encode_chunks_to_zmean(ae_model, tokenizer, chunks, device, max_len=max_len)  # [N,512]
            logits, aux = memrnn(M)

            # debug (optional)
            if random.random() < 0.001:
                print("gate_mean:", float(aux["gate_mean"].item()),
                      "attn_entropy:", float(aux["attn_entropy"].item()),
                      "num_chunks:", M.size(0))


            ce = F.cross_entropy(logits.unsqueeze(0), yi.unsqueeze(0))

            gate_reg = (aux["gate_mean"] - tau).pow(2)
            ent_reg = -aux["attn_entropy"]

            loss = ce + lam_gate * gate_reg + lam_ent * ent_reg
            loss_sum = loss_sum + loss
            used += 1

        if used == 0:
            continue

        loss_sum = loss_sum / used
        loss_sum.backward()
        torch.nn.utils.clip_grad_norm_(memrnn.parameters(), 1.0)
        opt.step()

        total_loss += float(loss_sum.item())
        n += 1

    return total_loss / max(1, n)

@torch.no_grad()
def eval_epoch(memrnn, ae_model, tokenizer, loader, device, max_chunks=16, max_len=144):
    memrnn.eval()
    total = 0
    correct = 0
    loss_total = 0.0
    n = 0

    for reviews, y in loader:
        y = y.to(device)

        batch_losses = []
        preds = []
        kept_labels = []  # <I FIXED THIS> keep labels aligned with preds when skipping reviews

        for review, yi in zip(reviews, y):
            chunks = review_to_chunks(review)
            if len(chunks) == 0:
                continue
            if len(chunks) > max_chunks:
                chunks = chunks[:max_chunks]

            M = encode_chunks_to_zmean(ae_model, tokenizer, chunks, device, max_len=max_len)
            logits, _ = memrnn(M)

            batch_losses.append(F.cross_entropy(logits.unsqueeze(0), yi.unsqueeze(0)))
            preds.append(int(torch.argmax(logits).item()))
            kept_labels.append(int(yi.item()))  # <I FIXED THIS>

        if len(preds) == 0:
            continue

        preds = torch.tensor(preds, device=device)
        yy = torch.tensor(kept_labels, device=device)  # <I FIXED THIS>

        loss = torch.stack(batch_losses).mean()
        loss_total += float(loss.item())
        n += 1

        correct += int((preds == yy).sum().item())
        total += int(yy.numel())

    return loss_total / max(1, n), (correct / max(1, total))


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.02, random_state=42, stratify=df["sentiment"])

train_ds = ReviewDataset(train_df)
val_ds   = ReviewDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_reviews, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, collate_fn=collate_reviews, num_workers=2)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

ae = BottleneckT5AE(
    model_name="t5-small",
    latent_dim=512,
    bottleneck_seq_len=32,
    pool_slots=16
).to(device)

ckpt = torch.load("/content/drive/MyDrive/AE model/best (1).pt", map_location=device)
ae.load_state_dict(ckpt)
ae.eval()

for p in ae.parameters():
    p.requires_grad = False

ae_model = ae
tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)

memrnn = ChunkMemRNN(latent_dim=LATENT_DIM, hidden_dim=256, topk=4, num_classes=2).to(device)
opt = torch.optim.AdamW(memrnn.parameters(), lr=2e-4, weight_decay=0.01)

EPOCHS = 3
for ep in range(1, EPOCHS + 1):
    tr = train_epoch(memrnn, ae_model, tokenizer, train_loader, opt, device,
                     max_chunks=16, max_len=MAX_LEN,
                     lam_gate=0.05, tau=0.20, lam_ent=0.01)
    vl, acc = eval_epoch(memrnn, ae_model, tokenizer, val_loader, device,
                         max_chunks=16, max_len=MAX_LEN)
    print(f"epoch {ep} | train_loss={tr:.4f} | val_loss={vl:.4f} | val_acc={acc:.4f}")

gate_mean: 0.49252843856811523 attn_entropy: 0.7605566382408142 num_chunks: 6
gate_mean: 0.492026150226593 attn_entropy: 0.4477543234825134 num_chunks: 4
gate_mean: 0.4886854290962219 attn_entropy: 0.6353898048400879 num_chunks: 5
gate_mean: 0.4879268407821655 attn_entropy: 0.4478861391544342 num_chunks: 4
gate_mean: 0.4973773956298828 attn_entropy: 0.0 num_chunks: 1
gate_mean: 0.48931536078453064 attn_entropy: 0.23069822788238525 num_chunks: 3
gate_mean: 0.48087459802627563 attn_entropy: 0.6355053186416626 num_chunks: 5
gate_mean: 0.4786863625049591 attn_entropy: 0.849908709526062 num_chunks: 7
gate_mean: 0.4839622378349304 attn_entropy: 0.23097380995750427 num_chunks: 3
gate_mean: 0.483795166015625 attn_entropy: 0.23091545701026917 num_chunks: 3
gate_mean: 0.4854585826396942 attn_entropy: 0.0 num_chunks: 2
gate_mean: 0.48164504766464233 attn_entropy: 0.23104503750801086 num_chunks: 3
gate_mean: 0.4950184226036072 attn_entropy: 0.0 num_chunks: 1
gate_mean: 0.47111520171165466 attn_ent

In [ ]:
# save memory RNN
torch.save(memrnn.state_dict(), "memrnn.pt")


Interpretation testing

In [ ]:
from transformers.modeling_outputs import BaseModelOutput

@torch.no_grad()
def decode_latent_vector_to_text(ae_model, tokenizer, v, max_new_tokens=80):
    """
    v: [D] where D=512 (same as LATENT_DIM after your normalize)
    We lift v into a pseudo decoder memory [1, M, d_model] and generate.
    """

    ae_model.eval()
    device = v.device
    D = v.shape[0]

    # <I FIXED THIS> Build "fake" encoder memory for T5 decoder
    Mlen = ae_model.bottleneck_seq_len          # 32
    d_model = ae_model.d_model                  # 512

    assert D == d_model, f"Expected {d_model}, got {D}"

    mem = v.view(1, 1, D).repeat(1, Mlen, 1)    # [1,32,512]
    mem = mem + ae_model.mem_slot_emb.unsqueeze(0)  # [1,32,512]

    mem_mask = torch.ones(1, Mlen, dtype=torch.long, device=device)

    enc_out = BaseModelOutput(last_hidden_state=mem)

    gen_ids = ae_model.t5.generate(
        encoder_outputs=enc_out,
        attention_mask=mem_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.15,
        early_stopping=True,
    )
    return tokenizer.decode(gen_ids[0], skip_special_tokens=True)

In [ ]:
class IntChunkMemRNN(nn.Module):
    def __init__(self, latent_dim=512, hidden_dim=256, topk=4, num_classes=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.topk = topk

        self.Wq = nn.Linear(hidden_dim, latent_dim, bias=False)
        self.Wg = nn.Linear(hidden_dim + latent_dim, latent_dim)

        # MUST match your checkpoint
        self.Wx = nn.Linear(latent_dim, hidden_dim, bias=False)
        self.Wh = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.Wm = nn.Linear(latent_dim, hidden_dim, bias=False)

        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, M, return_trace=False):
        device = M.device
        N, D = M.shape
        h = torch.zeros(self.hidden_dim, device=device)

        gate_means = []
        entropies = []

        trace = {
            "top_idx": [],
            "top_w": [],
            "gate_mean_t": [],
            "c_t": [],
            "x_t": [],
            "merged_t": [],
        } if return_trace else None

        for t in range(N):
            x_t = M[t]
            q = F.normalize(self.Wq(h), dim=-1)

            # causal retrieval (same as your trained code)
            if t == 0:
                c = torch.zeros(D, device=device)
                w = None
                idx = torch.tensor([-1], device=device)
            else:
                Mpast = M[:t]
                scores = (Mpast @ q)
                k = min(self.topk, t)
                topv, topi = torch.topk(scores, k=k, dim=0)
                w = torch.softmax(topv, dim=0)
                c = (w[:, None] * Mpast[topi]).sum(dim=0)
                idx = topi

            if w is None:
                entropies.append(torch.tensor(0.0, device=device))
            else:
                entropies.append(-(w * (w + 1e-8).log()).sum())

            g = torch.sigmoid(self.Wg(torch.cat([h, c], dim=0)))
            gate_means.append(g.mean())

            # exact trained update
            h = torch.tanh(self.Wx(x_t) + self.Wh(h) + self.Wm(g * c))

            if return_trace:
                trace["top_idx"].append(idx.detach().cpu())
                trace["top_w"].append(None if w is None else w.detach().cpu())
                trace["gate_mean_t"].append(float(g.mean().detach().cpu()))
                trace["c_t"].append(c.detach().cpu())
                trace["x_t"].append(x_t.detach().cpu())

                # <I FIXED THIS> store merged latent proxy for decoding
                merged = F.normalize((x_t + g.mean() * c).float(), dim=-1)
                trace["merged_t"].append(merged.detach().cpu())

        logits = self.classifier(h)
        aux = {
            "gate_mean": torch.stack(gate_means).mean() if gate_means else torch.tensor(0., device=device),
            "attn_entropy": torch.stack(entropies).mean() if entropies else torch.tensor(0., device=device),
        }

        if return_trace:
            return logits, aux, trace
        return logits, aux

In [ ]:
@torch.no_grad()
def interpret_one_review(review_text, label_str, ae_model, tokenizer, memrnn, device,
                         max_chunks=16, max_len=144, max_steps=8):
    chunks = review_to_chunks(review_text)
    if len(chunks) == 0:
        print("No chunks.")
        return
    chunks = chunks[:max_chunks]

    M = encode_chunks_to_zmean(ae_model, tokenizer, chunks, device, max_len=max_len)  # [N,512]
    logits, aux, trace = memrnn(M, return_trace=True)
    pred = int(torch.argmax(logits).item())

    print("True:", label_str, "| Pred:", pred, "| gate_mean:", float(aux["gate_mean"].item()),
          "| entropy:", float(aux["attn_entropy"].item()))
    print("="*80)

    N = M.size(0)
    steps = min(N, max_steps)

    for t in range(steps):
        x_t = trace["x_t"][t].to(device)
        c_t = trace["c_t"][t].to(device)
        gm  = trace["gate_mean_t"][t]

        # decode current chunk concept
        x_txt = decode_latent_vector_to_text(ae_model, tokenizer, x_t, max_new_tokens=60)

        # decode retrieved concept (if any)
        if t == 0:
            c_txt = "<no past memory>"
        else:
            c_txt = decode_latent_vector_to_text(ae_model, tokenizer, c_t, max_new_tokens=60)

        # merged proxy: x + gm*c
        merged = F.normalize((x_t + gm * c_t).float(), dim=-1)
        m_txt = decode_latent_vector_to_text(ae_model, tokenizer, merged, max_new_tokens=60)

        try:

          print(f"\n[t={t}] gate_mean_t={gm:.3f} | top_idx={trace['top_idx'][t].tolist()} | top_w={trace['top_w'][t].tolist()}")
          print("Chunk text:", chunks[t][:200])
          print("Decode(x_t):", x_txt[:250])
          print("Decode(c_t):", c_txt[:250])
          print("Decode(merged):", m_txt[:250])
        except:
          print(t)


In [ ]:
test_int_model = IntChunkMemRNN(latent_dim=512, hidden_dim=256, topk=4, num_classes=2).to(device)

mem_ckpt = torch.load("memrnn.pt", map_location=device)

# <I FIXED THIS> now the keys/shapes match because architecture matches the saved model
test_int_model.load_state_dict(mem_ckpt, strict=True)
test_int_model.eval()

IntChunkMemRNN(
  (Wq): Linear(in_features=256, out_features=512, bias=False)
  (Wg): Linear(in_features=768, out_features=512, bias=True)
  (Wx): Linear(in_features=512, out_features=256, bias=False)
  (Wh): Linear(in_features=256, out_features=256, bias=False)
  (Wm): Linear(in_features=512, out_features=256, bias=False)
  (classifier): Linear(in_features=256, out_features=2, bias=True)
)

In [ ]:
max_len = 0
max_idx = 0
for i,rev in enumerate(val_df.review.values):
  if len(rev) > max_len:
    max_len = len(rev)
    max_idx = i

In [ ]:
max_idx

432

In [ ]:
sample_text = val_df.iloc[432]["review"]
sample_label = val_df.iloc[432]["sentiment"]

interpret_one_review(
    sample_text,
    sample_label,
    ae_model,
    tokenizer,
    test_int_model,
    device
)


True: positive | Pred: 1 | gate_mean: 0.4119132161140442 | entropy: 1.1515493392944336
0

[t=1] gate_mean_t=0.399 | top_idx=[0] | top_w=[1.0]
Chunk text: And it's not "Laputa: Castle In The Sky", just "Castle In The Sky" for the dub, since Laputa is not such a nice word in Spanish (even though they use the word Laputa many times throughout the dub). Yo
Decode(x_t): In the morning, a young man has been able to speak English and speak English. Then he sat down with his wife and gave birth to a new baby.
Decode(c_t):  l'occasion de la délégation d'Isral, il a été décidé de mettre en uvre un projet de loisirs.
Decode(merged): :) This is the first time a man has been able to take part in a traditional sydney-style wedding. he drew a lot of inspiration from the early days of his life.

[t=2] gate_mean_t=0.405 | top_idx=[0, 1] | top_w=[0.5038091540336609, 0.4961908757686615]
Chunk text: And in my opinion, I think it's one of Miyazaki's best films with a powerful lesson tuckered inside this tw

In [ ]:
class ChunkMemRNN_NoMem(nn.Module):
    def __init__(self, latent_dim=512, hidden_dim=256, topk=4, num_classes=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.topk = topk

        self.Wq = nn.Linear(hidden_dim, latent_dim, bias=False)
        self.Wg = nn.Linear(hidden_dim + latent_dim, latent_dim)

        self.Wx = nn.Linear(latent_dim, hidden_dim, bias=False)
        self.Wh = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.Wm = nn.Linear(latent_dim, hidden_dim, bias=False)

        self.classifier = nn.Linear(hidden_dim, num_classes)

        self.disable_memory = True  # <I FIXED THIS>

    def forward(self, M):
        device = M.device
        N, D = M.shape
        h = torch.zeros(self.hidden_dim, device=device)

        gate_means = []
        entropies = []

        for t in range(N):
            x_t = M[t]

            # memory disabled
            c = torch.zeros(D, device=device)
            w = None

            entropies.append(torch.tensor(0.0, device=device))
            g = torch.sigmoid(self.Wg(torch.cat([h, c], dim=0)))
            gate_means.append(g.mean())

            h = torch.tanh(self.Wx(x_t) + self.Wh(h))  # NO memory term

        logits = self.classifier(h)
        aux = {
            "gate_mean": torch.stack(gate_means).mean(),
            "attn_entropy": torch.stack(entropies).mean(),
        }
        return logits, aux


In [ ]:
nomem_model = ChunkMemRNN_NoMem(
    latent_dim=512,
    hidden_dim=256,
    topk=4,
    num_classes=2
).to(device)

# load only matching weights
nomem_model.load_state_dict(
    torch.load("memrnn.pt", map_location=device),

)
nomem_model.eval()


ChunkMemRNN_NoMem(
  (Wq): Linear(in_features=256, out_features=512, bias=False)
  (Wg): Linear(in_features=768, out_features=512, bias=True)
  (Wx): Linear(in_features=512, out_features=256, bias=False)
  (Wh): Linear(in_features=256, out_features=256, bias=False)
  (Wm): Linear(in_features=512, out_features=256, bias=False)
  (classifier): Linear(in_features=256, out_features=2, bias=True)
)

In [ ]:
@torch.no_grad()
def eval_model(model, ae_model, tokenizer, loader, device,
               max_chunks=16, max_len=144):
    model.eval()
    correct = 0
    total = 0

    for reviews, y in loader:
        y = y.to(device)

        for review, yi in zip(reviews, y):
            chunks = review_to_chunks(review)
            if len(chunks) == 0:
                continue
            if len(chunks) > max_chunks:
                chunks = chunks[:max_chunks]

            M = encode_chunks_to_zmean(
                ae_model, tokenizer, chunks, device, max_len=max_len
            )

            logits, _ = model(M)
            pred = torch.argmax(logits).item()

            correct += int(pred == yi.item())
            total += 1

    return correct / max(1, total)

In [ ]:
acc_mem = eval_model(
    memrnn, ae_model, tokenizer, val_loader, device
)

acc_nomem = eval_model(
    nomem_model, ae_model, tokenizer, val_loader, device
)

print("===== MEMORY ABLATION =====")
print(f"With Memory    : {acc_mem:.4f}")
print(f"Zero Memory    : {acc_nomem:.4f}")
print(f"Δ Accuracy     : {acc_mem - acc_nomem:+.4f}")


===== MEMORY ABLATION =====
With Memory    : 0.8450
Zero Memory    : 0.8200
Δ Accuracy     : +0.0250


In [ ]:
class TraceableMemRNN(nn.Module):
    def __init__(self, latent_dim=512, hidden_dim=256, topk=4, num_classes=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.topk = topk

        self.Wq = nn.Linear(hidden_dim, latent_dim, bias=False)
        self.Wg = nn.Linear(hidden_dim + latent_dim, latent_dim)

        self.Wx = nn.Linear(latent_dim, hidden_dim, bias=False)
        self.Wh = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.Wm = nn.Linear(latent_dim, hidden_dim, bias=False)

        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, M, return_trace=False):  # <I FIXED THIS>
        """
        M: [N,D] normalized chunk latents for ONE review
        return_trace: if True, returns (logits, aux, trace)
        """
        device = M.device
        N, D = M.shape
        h = torch.zeros(self.hidden_dim, device=device)

        gate_means = []
        entropies = []

        trace = None
        if return_trace:  # <I FIXED THIS>
            trace = {
                "x_t_idx": [],        # timestep indices
                "top_idx": [],        # list of tensors [k]
                "top_w": [],          # list of tensors [k]
                "top_scores": [],     # list of tensors [k] (cos/dot scores used before softmax)
                "gate_mean_t": [],    # list of floats
                "N": N
            }

        for t in range(N):
            x_t = M[t]

            if t == 0:
                c = torch.zeros(D, device=device)
                w = None
                topi = torch.tensor([], device=device, dtype=torch.long)
                topv = torch.tensor([], device=device)
            else:
                Mpast = M[:t]  # causal: only past

                q = F.normalize(self.Wq(h), dim=-1)  # [D]
                scores = Mpast @ q                   # [t]  (dot == cosine because Mpast and q are normalized)

                k = min(self.topk, t)
                topv, topi = torch.topk(scores, k=k, dim=0)  # [k], [k]
                w = torch.softmax(topv, dim=0)               # [k]
                c = (w[:, None] * Mpast[topi]).sum(dim=0)    # [D]

            # entropy always defined
            if w is None:
                entropies.append(torch.tensor(0.0, device=device))
            else:
                entropies.append(-(w * (w + 1e-8).log()).sum())

            g = torch.sigmoid(self.Wg(torch.cat([h, c], dim=0)))  # [D]
            gate_means.append(g.mean())

            # true RNN update
            h = torch.tanh(self.Wx(x_t) + self.Wh(h) + self.Wm(g * c))

            if return_trace:  # <I FIXED THIS>
                trace["x_t_idx"].append(t)
                trace["top_idx"].append(topi.detach().cpu())
                trace["top_w"].append((w.detach().cpu() if w is not None else torch.tensor([])))
                trace["top_scores"].append(topv.detach().cpu())
                trace["gate_mean_t"].append(float(g.mean().detach().cpu()))

        logits = self.classifier(h)
        aux = {
            "gate_mean": torch.stack(gate_means).mean() if gate_means else torch.tensor(0., device=device),
            "attn_entropy": torch.stack(entropies).mean() if entropies else torch.tensor(0., device=device),
        }

        if return_trace:
            return logits, aux, trace  # <I FIXED THIS>
        return logits, aux


@torch.no_grad()
def run_and_trace_one_review(mem_model, ae_model, tokenizer, review_text, device,
                             max_chunks=16, max_len=144, topk_print=4):
    """
    Prints:
      - each chunk (t)
      - which past chunks were retrieved (indices + weights + scores)
      - gate_mean_t
    """
    mem_model.eval()
    ae_model.eval()

    chunks = review_to_chunks(review_text)
    if len(chunks) == 0:
        print("No chunks produced for this review.")
        return

    if len(chunks) > max_chunks:
        chunks = chunks[:max_chunks]

    # encode chunks -> M
    M = encode_chunks_to_zmean(ae_model, tokenizer, chunks, device, max_len=max_len)  # [N,512]

    logits, aux, trace = mem_model(M, return_trace=True)
    pred = int(torch.argmax(logits).item())

    print(f"Pred class: {pred} | gate_mean(avg)={float(aux['gate_mean']):.4f} | entropy(avg)={float(aux['attn_entropy']):.4f}")
    print("=" * 90)

    N = trace["N"]
    for t in range(N):
        gm = trace["gate_mean_t"][t]
        topi = trace["top_idx"][t].tolist()
        topw = trace["top_w"][t].tolist()
        tops = trace["top_scores"][t].tolist()

        print(f"\n[t={t}] gate_mean_t={gm:.3f}")
        print("Chunk text:", chunks[t][:300])

        if t == 0:
            print("Retrieved: (none; no past)")
            continue

        k = min(topk_print, len(topi))
        print("Retrieved (past):")
        for j in range(k):
            idx = topi[j]
            wj = topw[j]
            sj = tops[j]
            print(f"  - idx={idx:>2d} | w={wj:.3f} | score={sj:.3f} | text={chunks[idx][:160]}")

    return pred, aux, trace


In [ ]:
import re
from collections import Counter

def _tok(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    toks = [t for t in s.split() if len(t) > 2]
    return toks

def top_word_overlap(a, b, k=8):
    A = Counter(_tok(a))
    B = Counter(_tok(b))
    inter = (A & B)
    return [w for w,_ in inter.most_common(k)]

@torch.no_grad()
def run_and_trace_one_review_step2plus(mem_model, ae_model, tokenizer, review_text, device,
                                       max_chunks=16, max_len=144,
                                       topk_print=4, top_only=False):
    mem_model.eval()
    ae_model.eval()

    chunks = review_to_chunks(review_text)
    if len(chunks) == 0:
        print("No chunks produced for this review.")
        return
    if len(chunks) > max_chunks:
        chunks = chunks[:max_chunks]

    M = encode_chunks_to_zmean(ae_model, tokenizer, chunks, device, max_len=max_len)  # [N,D]

    # --- we recompute scores here to get stats over all past (without changing model) ---
    logits, aux, trace = mem_model(M, return_trace=True)

    pred = int(torch.argmax(logits).item())
    print(f"Pred class: {pred} | gate_mean(avg)={float(aux['gate_mean']):.4f} | entropy(avg)={float(aux['attn_entropy']):.4f}")
    print("=" * 90)

    N = trace["N"]
    for t in range(N):
        gm = trace["gate_mean_t"][t]
        print(f"\n[t={t}] gate_mean_t={gm:.3f}")
        print("Chunk text:", chunks[t][:260])

        if t == 0:
            print("Retrieved: (none; no past)")
            continue

        # score stats over *all past* using same query definition as the model
        # We need h, but trace doesn't store it; so we compute approximate stats from stored top scores only.
        # So: show top scores spread as a proxy.
        tops = trace["top_scores"][t]
        if len(tops) > 0:
            print(f"Top-score stats: min={float(tops.min()):.4f} max={float(tops.max()):.4f} mean={float(tops.mean()):.4f}")

        topi = trace["top_idx"][t].tolist()
        topw = trace["top_w"][t].tolist()
        tops = trace["top_scores"][t].tolist()

        if len(topi) == 0:
            print("Retrieved: (none)")
            continue

        if top_only:
            topi = topi[:1]; topw = topw[:1]; tops = tops[:1]

        print("Retrieved (past):")
        for j in range(min(topk_print, len(topi))):
            idx = topi[j]
            wj = topw[j]
            sj = tops[j]
            overlap = top_word_overlap(chunks[t], chunks[idx], k=8)
            print(f"  - idx={idx:>2d} | w={wj:.3f} | score={sj:.3f} | overlap={overlap}")
            print(f"    past text: {chunks[idx][:170]}")

    return pred, aux, trace


In [ ]:
trace_rnn = TraceableMemRNN().to(device)

trace_ckpt = torch.load("memrnn.pt", map_location=device)

trace_rnn.load_state_dict(trace_ckpt)
trace_rnn.eval()

TraceableMemRNN(
  (Wq): Linear(in_features=256, out_features=512, bias=False)
  (Wg): Linear(in_features=768, out_features=512, bias=True)
  (Wx): Linear(in_features=512, out_features=256, bias=False)
  (Wh): Linear(in_features=256, out_features=256, bias=False)
  (Wm): Linear(in_features=512, out_features=256, bias=False)
  (classifier): Linear(in_features=256, out_features=2, bias=True)
)

In [ ]:
max_len = 0
max_idx = 0
for i,rev in enumerate(df.review.values):
  if len(rev) > max_len:
    max_len = len(rev)
    max_idx = i

In [ ]:
# ===== TEST 1: GLOBAL RETRIEVAL DIAGNOSTIC (original vs Step-4-fixed) =====
# Uses YOUR existing vars: df, trace_rnn, ae_model, tokenizer, device
# Does NOT change any of your var names.

import random, math
import torch
import torch.nn.functional as F
from collections import Counter

@torch.no_grad()
def run_and_trace_one_review_step4_fixidx0(
    trace_rnn, ae_model, tokenizer,
    review_text, device,
    max_chunks=16, max_len=144,
    top_only=True,
    forbid_idx0=True,
    recency_alpha=0.0,   # try 0.2 later if you want
):
    """
    Same idea as your step2plus tracer, but with an inference-time fix:
      - optionally forbid retrieving idx=0 for t>0
      - optional recency bias (favor recent past chunks)
    Returns: pred(int), aux(dict-like), trace(dict)
    """
    trace_rnn.eval()
    ae_model.eval()

    chunks = review_to_chunks(review_text)
    if len(chunks) == 0:
        return None

    if len(chunks) > max_chunks:
        chunks = chunks[:max_chunks]

    M = encode_chunks_to_zmean(ae_model, tokenizer, chunks, device, max_len=max_len)  # [N,D]
    N, D = M.shape

    # ---- replicate forward with tracing + fix ----
    h = torch.zeros(trace_rnn.hidden_dim, device=device)

    gate_means = []
    entropies = []

    trace = {
        "top_idx": [],
        "top_w": [],
        "scores_top": [],   # raw top scores
        "gate_mean_t": [],
        "chunk_text": chunks,
    }

    for t in range(N):
        x_t = M[t]

        if t == 0:
            c = torch.zeros(D, device=device)
            w = torch.tensor([1.0], device=device)
            idx = torch.tensor([-1], device=device)
            top_scores = torch.tensor([0.0], device=device)
        else:
            Mpast = M[:t]  # [t,D]
            q = F.normalize(trace_rnn.Wq(h), dim=-1)     # [D]
            scores = (Mpast @ q)                         # [t]

            # Step-4 fix A: forbid always pulling from idx=0
            if forbid_idx0 and t > 0:
                scores = scores.clone()
                scores[0] = -1e9

            # Step-4 fix B: slight recency bias (optional)
            # higher index => more recent
            if recency_alpha != 0.0:
                rec = torch.arange(t, device=device, dtype=scores.dtype)
                rec = rec / max(1, t-1)
                scores = scores + recency_alpha * rec

            k = min(trace_rnn.topk, t)
            topv, topi = torch.topk(scores, k=k, dim=0)
            w = torch.softmax(topv, dim=0)
            c = (w[:, None] * Mpast[topi]).sum(dim=0)
            idx = topi
            top_scores = topv

        # gate + update exactly like your ChunkMemRNN
        g = torch.sigmoid(trace_rnn.Wg(torch.cat([h, c], dim=0)))
        gate_means.append(g.mean())

        # entropy
        if idx.numel() == 1 and idx[0].item() == -1:
            entropies.append(torch.tensor(0.0, device=device))
        else:
            entropies.append(-(w * (w + 1e-8).log()).sum())

        h = torch.tanh(trace_rnn.Wx(x_t) + trace_rnn.Wh(h) + trace_rnn.Wm(g * c))

        trace["top_idx"].append(idx.detach().cpu())
        trace["top_w"].append(w.detach().cpu())
        trace["scores_top"].append(top_scores.detach().cpu())
        trace["gate_mean_t"].append(float(g.mean().detach().cpu()))

    logits = trace_rnn.classifier(h)
    pred = int(torch.argmax(logits).item())
    aux = {
        "gate_mean": float(torch.stack(gate_means).mean().detach().cpu()) if gate_means else 0.0,
        "attn_entropy": float(torch.stack(entropies).mean().detach().cpu()) if entropies else 0.0,
    }
    return pred, aux, trace


@torch.no_grad()
def global_retrieval_diagnostic(
    df, trace_rnn, ae_model, tokenizer, device,
    n=30, max_chunks=16, max_len=144,
    use_step4_fix=False,
    forbid_idx0=True,
    recency_alpha=0.0,
    seed=42
):
    random.seed(seed)
    idxs = random.sample(range(len(df)), k=min(n, len(df)))

    top1_is0 = []
    top1_counter = Counter()
    allidx_counter = Counter()
    ent_list = []
    gate_list = []
    examples_collapse = []
    examples_not = []

    for di in idxs:
        review_text = df["review"].iloc[di]

        if use_step4_fix:
            out = run_and_trace_one_review_step4_fixidx0(
                trace_rnn, ae_model, tokenizer, review_text, device,
                max_chunks=max_chunks, max_len=max_len,
                forbid_idx0=forbid_idx0,
                recency_alpha=recency_alpha,
            )
        else:
            # your existing function
            pred, aux, trace = run_and_trace_one_review_step2plus(
                trace_rnn, ae_model, tokenizer,
                review_text=review_text,
                device=device,
                max_chunks=max_chunks,
                top_only=True
            )
            out = (pred, {"gate_mean": float(aux["gate_mean"]), "attn_entropy": float(aux["attn_entropy"])}, trace)

        if out is None:
            continue

        pred, aux, trace = out
        gate_list.append(aux["gate_mean"])
        ent_list.append(aux["attn_entropy"])

        # compute "frac(top1==0)" over timesteps with past
        hits = 0
        steps = 0
        for t, idx_t in enumerate(trace["top_idx"]):
            idx_t = idx_t.numpy().tolist()
            if len(idx_t) == 0:
                continue
            if idx_t[0] == -1:
                continue
            steps += 1
            top1 = idx_t[0]
            top1_counter[top1] += 1
            if top1 == 0:
                hits += 1

            for j in idx_t:
                if j != -1:
                    allidx_counter[j] += 1

        frac0 = (hits / max(1, steps))
        top1_is0.append(frac0)

        if steps > 0:
            if frac0 >= 0.9:
                examples_collapse.append((di, frac0, steps, aux["attn_entropy"], aux["gate_mean"]))
            if frac0 <= 0.35:
                examples_not.append((di, frac0, steps, aux["attn_entropy"], aux["gate_mean"]))

    print("===== GLOBAL MEMORY DIAGNOSTIC (random sample) =====")
    print("Use step4 fix            :", use_step4_fix, "| forbid_idx0:", forbid_idx0, "| recency_alpha:", recency_alpha)
    print("N samples                :", len(top1_is0))
    print("Mean frac(top1==0)        :", float(sum(top1_is0)/max(1,len(top1_is0))))
    print("Mean attn_entropy(avg)    :", float(sum(ent_list)/max(1,len(ent_list))))
    print("Mean gate_mean(avg)       :", float(sum(gate_list)/max(1,len(gate_list))))
    print()
    print("Top-1 retrieved indices (global):")
    for k,v in top1_counter.most_common(10):
        print(f"  idx={k:>3d} | count={v}")
    print()
    print("All retrieved indices across top-k (global):")
    for k,v in allidx_counter.most_common(10):
        print(f"  idx={k:>3d} | count={v}")
    print()
    print("Top reviews where retrieval collapses to idx=0:")
    for row in examples_collapse[:5]:
        di, frac0, steps, ent, gate = row
        print(f"  df_idx={di} | frac0={frac0:.3f} | steps={steps} | ent={ent:.3f} | gate={gate:.3f}")
    print()
    print("Top reviews where retrieval is NOT dominated by idx=0:")
    for row in examples_not[:5]:
        di, frac0, steps, ent, gate = row
        print(f"  df_idx={di} | frac0={frac0:.3f} | steps={steps} | ent={ent:.3f} | gate={gate:.3f}")
    return {
        "mean_frac0": float(sum(top1_is0)/max(1,len(top1_is0))),
        "top1_counter": top1_counter,
        "allidx_counter": allidx_counter
    }

# Run original vs Step-4-fixed
orig_stats = global_retrieval_diagnostic(df, trace_rnn, ae_model, tokenizer, device, n=30, use_step4_fix=False)
fix_stats  = global_retrieval_diagnostic(df, trace_rnn, ae_model, tokenizer, device, n=30, use_step4_fix=True, forbid_idx0=True, recency_alpha=0.13)


Pred class: 0 | gate_mean(avg)=0.4149 | entropy(avg)=0.4468

[t=0] gate_mean_t=0.479
Chunk text: Water shows the plight of Indian widows in the late 1930s, says in the end that the problem still exists largely by giving statistics in the end, refers to Gandhi several times in the movie before finally having a scene depicting him and does nothing extra ord
Retrieved: (none; no past)

[t=1] gate_mean_t=0.395
Chunk text: What if a movie is made on racism in America in a particular year which ends with 'x number of Americans still experience racism today'. a) How would it be relevant, and, b) How would it be some thing so extra ordinary being depicted in cinema. A view I read f
Top-score stats: min=0.1443 max=0.1443 mean=0.1443
Retrieved (past):
  - idx= 0 | w=1.000 | score=0.144 | overlap=['the', 'movie', 'still', 'and', 'extra', 'that', 'for', 'problem']
    past text: Water shows the plight of Indian widows in the late 1930s, says in the end that the problem still exists largely by givi

In [ ]:
# ===== TEST 2: ACCURACY (original vs Step-4-fixed inference) =====
# Uses YOUR existing vars: trace_rnn, test_loader, ae_model, tokenizer, device
# No retraining; just inference-time accuracy.

@torch.no_grad()
def eval_epoch_mem(model, ae_model, tokenizer, loader, device, max_chunks=16, max_len=144):
    model.eval()
    total = 0
    correct = 0

    for reviews, y in loader:
        y = y.to(device)

        preds = []
        kept = []

        for review, yi in zip(reviews, y):
            chunks = review_to_chunks(review)
            if len(chunks) == 0:
                continue
            if len(chunks) > max_chunks:
                chunks = chunks[:max_chunks]

            M = encode_chunks_to_zmean(ae_model, tokenizer, chunks, device, max_len=max_len)
            logits, aux = model(M)
            preds.append(int(torch.argmax(logits).item()))
            kept.append(int(yi.item()))

        if len(preds) == 0:
            continue

        preds = torch.tensor(preds, device=device)
        kept = torch.tensor(kept, device=device)
        correct += int((preds == kept).sum().item())
        total += int(kept.numel())

    return correct / max(1, total)


@torch.no_grad()
def eval_epoch_mem_step4_fixidx0(trace_rnn, ae_model, tokenizer, loader, device,
                                max_chunks=16, max_len=144, forbid_idx0=True, recency_alpha=0.0):
    total = 0
    correct = 0

    for reviews, y in loader:
        y = y.to(device)

        preds = []
        kept = []

        for review, yi in zip(reviews, y):
            chunks = review_to_chunks(review)
            if len(chunks) == 0:
                continue
            if len(chunks) > max_chunks:
                chunks = chunks[:max_chunks]

            # run step4 forward (same as tracer but without printing)
            M = encode_chunks_to_zmean(ae_model, tokenizer, chunks, device, max_len=max_len)
            N, D = M.shape
            h = torch.zeros(trace_rnn.hidden_dim, device=device)

            for t in range(N):
                x_t = M[t]
                if t == 0:
                    c = torch.zeros(D, device=device)
                else:
                    Mpast = M[:t]
                    q = F.normalize(trace_rnn.Wq(h), dim=-1)
                    scores = (Mpast @ q)

                    if forbid_idx0 and t > 0:
                        scores = scores.clone()
                        scores[0] = -1e9

                    if recency_alpha != 0.0:
                        rec = torch.arange(t, device=device, dtype=scores.dtype)
                        rec = rec / max(1, t-1)
                        scores = scores + recency_alpha * rec

                    k = min(trace_rnn.topk, t)
                    topv, topi = torch.topk(scores, k=k, dim=0)
                    w = torch.softmax(topv, dim=0)
                    c = (w[:, None] * Mpast[topi]).sum(dim=0)

                g = torch.sigmoid(trace_rnn.Wg(torch.cat([h, c], dim=0)))
                h = torch.tanh(trace_rnn.Wx(x_t) + trace_rnn.Wh(h) + trace_rnn.Wm(g * c))

            logits = trace_rnn.classifier(h)
            preds.append(int(torch.argmax(logits).item()))
            kept.append(int(yi.item()))

        if len(preds) == 0:
            continue

        preds = torch.tensor(preds, device=device)
        kept = torch.tensor(kept, device=device)
        correct += int((preds == kept).sum().item())
        total += int(kept.numel())

    return correct / max(1, total)

acc_orig = eval_epoch_mem(trace_rnn, ae_model, tokenizer, val_loader, device)
acc_fix  = eval_epoch_mem_step4_fixidx0(trace_rnn, ae_model, tokenizer, val_loader, device,
                                        forbid_idx0=True, recency_alpha=0.13)

print("===== ACCURACY (inference-time) =====")
print(f"Original         : {acc_orig:.4f}")
print(f"Step4 forbid idx0: {acc_fix:.4f}")
print(f"Δ Accuracy       : {acc_fix - acc_orig:+.4f}")


===== ACCURACY (inference-time) =====
Original         : 0.8450
Step4 forbid idx0: 0.8420
Δ Accuracy       : -0.0030


In [ ]:
@torch.no_grad()
def interpret_one_review(review_text, label_str, ae_model, tokenizer, memrnn, device,
                         max_chunks=16, max_len=144, max_steps=8):
    chunks = review_to_chunks(review_text)
    if len(chunks) == 0:
        print("No chunks.")
        return
    chunks = chunks[:max_chunks]

    M = encode_chunks_to_zmean(ae_model, tokenizer, chunks, device, max_len=max_len)  # [N,512]
    logits, aux, trace = memrnn(M, return_trace=True)
    pred = int(torch.argmax(logits).item())

    print("True:", label_str, "| Pred:", pred, "| gate_mean:", float(aux["gate_mean"].item()),
          "| entropy:", float(aux["attn_entropy"].item()))
    print("="*80)

    N = M.size(0)
    steps = min(N, max_steps)

    for t in range(steps):
        x_t = trace["x_t"][t].to(device)
        c_t = trace["c_t"][t].to(device)
        gm  = trace["gate_mean_t"][t]

        # decode current chunk concept
        x_txt = decode_latent_vector_to_text(ae_model, tokenizer, x_t, max_new_tokens=60)

        # decode retrieved concept (if any)
        if t == 0:
            c_txt = "<no past memory>"
        else:
            c_txt = decode_latent_vector_to_text(ae_model, tokenizer, c_t, max_new_tokens=60)

        # merged proxy: x + gm*c
        merged = F.normalize((x_t + gm * c_t).float(), dim=-1)
        m_txt = decode_latent_vector_to_text(ae_model, tokenizer, merged, max_new_tokens=60)

        print(f"\n[t={t}] gate_mean_t={gm:.3f} | top_idx={trace['top_idx'][t].tolist()} | top_w={trace['top_w'][t].tolist()}")
        print("Chunk text:", chunks[t][:200])
        print("Decode(x_t):", x_txt[:250])
        print("Decode(c_t):", c_txt[:250])
        print("Decode(merged):", m_txt[:250])


In [ ]:
class NoMemoryRNN(nn.Module):
    def __init__(self, latent_dim=512, hidden_dim=256, num_classes=2, dropout_p=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim

        # <I FIXED THIS> Use GRUCell as a stronger, standard RNN baseline (still no memory)
        self.rnn = nn.GRUCell(input_size=latent_dim, hidden_size=hidden_dim)

        # <I FIXED THIS> Regularization that helps stability / generalization
        self.dropout = nn.Dropout(dropout_p)

        # <I FIXED THIS> Classifier on pooled representation (stronger than only last h)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, M):
        """
        M: [N, D] (normalized chunk latents for ONE review)
        returns: logits [C]
        """
        device = M.device
        N, D = M.shape

        # handle edge case safely
        if N == 0:
            return torch.zeros(self.classifier.out_features, device=device)

        h = torch.zeros(self.hidden_dim, device=device)
        hs = []

        for t in range(N):
            x_t = M[t]  # [D]

            # <I FIXED THIS> GRUCell expects [D] input OK, but ensure consistent dtype
            h = self.rnn(x_t, h)  # [H]
            hs.append(h)

        H = torch.stack(hs, dim=0)  # [N, H]

        # <I FIXED THIS> Mean pooling over time (strong baseline, avoids "last state only" weakness)
        h_pool = H.mean(dim=0)  # [H]
        h_pool = self.dropout(h_pool)

        logits = self.classifier(h_pool)  # [C]
        return logits


In [ ]:
def train_epoch_nomem(model, ae_model, tokenizer, loader, opt, device,
                      max_chunks=16, max_len=144):
    model.train()
    total_loss = 0.0
    n_batches = 0

    for reviews, y in loader:
        y = y.to(device)
        opt.zero_grad(set_to_none=True)

        loss_sum = 0.0
        used = 0

        for review, yi in zip(reviews, y):
            chunks = review_to_chunks(review)
            if len(chunks) == 0:
                continue

            if len(chunks) > max_chunks:
                chunks = chunks[:max_chunks]

            M = encode_chunks_to_zmean(ae_model, tokenizer, chunks, device, max_len=max_len)  # [N,512]
            logits = model(M)  # <I FIXED THIS> model returns logits only

            ce = F.cross_entropy(logits.unsqueeze(0), yi.unsqueeze(0))
            loss_sum = loss_sum + ce
            used += 1

        if used == 0:
            continue

        loss = loss_sum / used
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        total_loss += float(loss.item())
        n_batches += 1

    return total_loss / max(1, n_batches)


@torch.no_grad()
def eval_epoch_nomem(model, ae_model, tokenizer, loader, device,
                     max_chunks=16, max_len=144):
    model.eval()
    total = 0
    correct = 0
    loss_total = 0.0
    n_batches = 0

    for reviews, y in loader:
        y = y.to(device)

        batch_losses = []
        preds = []
        kept_labels = []  # <I FIXED THIS> keep labels aligned with preds when skipping reviews

        for review, yi in zip(reviews, y):
            chunks = review_to_chunks(review)
            if len(chunks) == 0:
                continue

            if len(chunks) > max_chunks:
                chunks = chunks[:max_chunks]

            M = encode_chunks_to_zmean(ae_model, tokenizer, chunks, device, max_len=max_len)
            logits = model(M)  # <I FIXED THIS> logits only

            batch_losses.append(F.cross_entropy(logits.unsqueeze(0), yi.unsqueeze(0)))
            preds.append(int(torch.argmax(logits).item()))
            kept_labels.append(int(yi.item()))  # <I FIXED THIS>

        if len(preds) == 0:
            continue

        preds = torch.tensor(preds, device=device)
        yy = torch.tensor(kept_labels, device=device)  # <I FIXED THIS>

        loss = torch.stack(batch_losses).mean()
        loss_total += float(loss.item())
        n_batches += 1

        correct += int((preds == yy).sum().item())
        total += int(yy.numel())

    return loss_total / max(1, n_batches), (correct / max(1, total))

In [ ]:
model = NoMemoryRNN(latent_dim=512, hidden_dim=256, num_classes=2, dropout_p=0.1).to(device)
opt2 = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)

best_val_acc = -1.0
best_path = "best_nomem_gru.pt"  # <I FIXED THIS> save best baseline

for ep in range(1, EPOCHS + 1):
    tr_loss = train_epoch_nomem(
        model=model,
        ae_model=ae_model,
        tokenizer=tokenizer,
        loader=train_loader,
        opt=opt2,
        device=device,
        max_chunks=16,
        max_len=MAX_LEN
    )

    val_loss, val_acc = eval_epoch_nomem(
        model=model,
        ae_model=ae_model,
        tokenizer=tokenizer,
        loader=val_loader,
        device=device,
        max_chunks=16,
        max_len=MAX_LEN
    )

    print(f"epoch {ep} | train_loss={tr_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

    # <I FIXED THIS> checkpointing best model by val_acc
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        import os
        os.makedirs("NoMemRNN", exist_ok=True)
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "opt_state_dict": opt2.state_dict(),
                "epoch": ep,
                "val_acc": float(val_acc),
                "latent_dim": 512,
                "hidden_dim": 256,
                "num_classes": 2,
                "dropout_p": 0.1,
            },
            best_path
        )
        print(f"Saved best NoMemoryRNN to: {best_path} (val_acc={best_val_acc:.4f})")


epoch 1 | train_loss=0.3852 | val_loss=0.3358 | val_acc=0.8430
Saved best NoMemoryRNN to: best_nomem_gru.pt (val_acc=0.8430)
epoch 2 | train_loss=0.3368 | val_loss=0.3350 | val_acc=0.8470
Saved best NoMemoryRNN to: best_nomem_gru.pt (val_acc=0.8470)
epoch 3 | train_loss=0.3262 | val_loss=0.3184 | val_acc=0.8550
Saved best NoMemoryRNN to: best_nomem_gru.pt (val_acc=0.8550)
